In [ ]:
import pandas as pd
from dataclasses import dataclass
from datetime import datetime

In [ ]:


from collections import defaultdict


@dataclass
class ClientMissingData:
    missing_dates: int
    zeros_count: int

def process_clients(
        month_df: pd.DataFrame, 
        year: int, 
        month: int,
        batch_size: int = 100
) -> dict[str, ClientMissingData]:
    missing_data_dict : dict[str, ClientMissingData] = defaultdict(lambda: ClientMissingData(missing_dates=0, zeros_count=0))
    all_ids = month_df['id'].unique()
    id_batches = [all_ids[i:i + batch_size] for i in range(0, len(all_ids), batch_size)]

    data_points_total = 24 * pd.Period(f'{year}-{month}').days_in_month
    index = 0
    for batch_ids in id_batches:
        batch_data = month_df[month_df['id'].isin(batch_ids)]
        # We have to verify there's data for each day and hour as we know the count
        missing_entries = batch_data.groupby('id').size().reset_index(name='counts')
        zero_values = batch_data[batch_data['valor'] == 0].groupby('id').size().reset_index(name='zeros_count')
        for client_id in batch_ids:
            client_missing_entries = missing_entries[missing_entries['id'] == client_id]
            if client_missing_entries.empty:
                client_missing_entries_count = 0
            else:
                client_missing_entries_count = data_points_total - int(client_missing_entries['counts'].iloc[0])
            client_zeros = zero_values[zero_values['id'] == client_id]
            if client_zeros.empty:
                client_zeros = 0
            else:
                client_zeros = int(client_zeros['zeros_count'].iloc[0])
            if client_missing_entries_count > 0 or client_zeros > 0:
                missing_data_dict[client_id] = ClientMissingData(
                    missing_dates=client_missing_entries_count,
                    zeros_count=client_zeros
                )
            del client_missing_entries
            del client_zeros
            index += 1
        del batch_data
        del missing_entries
        del zero_values
        if index % 100 == 0:
            print(f"Processed {index} clients...")
            print(f"Amount of clients with missing data so far: {len(missing_data_dict)}")
    return missing_data_dict

In [ ]:
base_path = '~/ORT/Tesis/ute/raw/anii_residenciales/year=2024/month=4/'
df1 = pd.read_parquet(f'{base_path}part-00034-0cf868e2-2028-4bec-ad81-c195cae397e0.c000.snappy.parquet')
df2 = pd.read_parquet(f'{base_path}part-00127-0cf868e2-2028-4bec-ad81-c195cae397e0.c000.snappy.parquet')
df_concat = pd.concat([df1, df2])

In [ ]:
results = process_clients(df_concat, 2024, 4, batch_size=100)
print(f"Processed {len(results)} clients.")

In [ ]:
# detect how many clients have missing data

missing_dates = {k: v for k, v in results.items() if v.missing_dates > 0}
print(f"Clients with missing dates: {len(missing_dates)}")
zeros = {k: v for k, v in results.items() if v.zeros_count > 0}
print(f"Clients with zero values: {len(zeros)}")
missing_dates_and_zeros = {k: v for k, v in results.items() if v.missing_dates > 0 and v.zeros_count > 0}
print(f"Clients with missing dates and zero values: {len(missing_dates_and_zeros)}")